In [ ]:
import os
import numpy as np
import pandas as pd

DATA_PATH = os.path.join('CharRecog','Data','emnist')
def get_data(filename,path=DATA_PATH):
    file_path = path+filename
    df = pd.read_csv(file_path,header=None)
    data = df.iloc[:,1:]
    target = df.iloc[:,0]
    del df
    return target,data
    

In [ ]:
print("Loading Traning Data ...")
train_label,train_data = get_data('/emnist-mnist-train.csv')
test_label,test_data = get_data('/emnist-mnist-test.csv')
print(train_label.head())
print(train_data.head())

In [ ]:
train_label.value_counts()

In [ ]:
# The merged classes, as suggested by the NIST, are for the letters C, I, J, K, L, M, O, P, S, U,V, W, X, Y and Z
def getlabeldict(file,path = DATA_PATH):
    lbl_dict = {}
    with open(path+file) as f:
        for line in f.readlines():
            line = line.split(' ')
            lbl_dict[line[0]] = str(chr(int(line[1])))
    return lbl_dict


In [ ]:
lbl_dict = getlabeldict('/emnist-mnist-mapping.txt')
print(lbl_dict.items())

In [ ]:
test_data.info()

In [ ]:
def rotate(image):
    image = image.reshape([28, 28])
    image = image = image.transpose()
    return image.reshape([28 * 28])
train_data = np.apply_along_axis(rotate,1,train_data)
test_data = np.apply_along_axis(rotate,1,test_data)

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
def plotImage(n):
    print(lbl_dict[str(train_label.values[n])])
    image = train_data[n].reshape([28, 28])
    plt.imshow(image, cmap = mpl.cm.binary)
    plt.show()
plotImage(12)

In [ ]:
plt.imshow(test_data[32].reshape([28, 28]), cmap = mpl.cm.binary)
plt.show()
print(lbl_dict[str(test_label.values[32])])

In [ ]:
y_train_5 = (train_label == 5)
y_test_5 = (test_label == 5)

In [ ]:
from sklearn.linear_model import SGDClassifier
sgd_clf = SGDClassifier(random_state=42)
sgd_clf.fit(train_data, y_train_5)

In [ ]:
# sgd_clf.predict(test_data[15])
for i in range(100):
    if sgd_clf.predict([train_data[i]]) == True:
        print(i)

In [ ]:
sgd_clf.predict([train_data[10]])

In [ ]:
print(lbl_dict[str(train_label.values[10])])
plt.imshow(train_data[10].reshape([28, 28]), cmap = mpl.cm.binary)
plt.show()

In [ ]:
# save the Model
from joblib import dump, load
MODEL_PATH = os.path.join('CharRecog','test','Model')
def saveModel(model,filename):
    dump(model, os.path.join(MODEL_PATH,filename+'.joblib')) # 'filename.joblib'

def loadModel(filename):
    return load(os.path.join(MODEL_PATH,filename+'.joblib')) # 'filename.joblib'
saveModel(sgd_clf,"Model_Linear_SGD_IS_FIVE")
sgd_model = None
sgd_clf = loadModel("Model_Linear_SGD_IS_FIVE")

In [ ]:
from sklearn.model_selection import cross_val_score
cross_val_score(sgd_clf, train_data, y_train_5, cv=3, scoring="accuracy")

In [ ]:
from sklearn.model_selection import cross_val_predict
y_train_pred = cross_val_predict(sgd_clf, train_data, y_train_5, cv=3)

In [ ]:
from sklearn.metrics import confusion_matrix
confusion_matrix(y_train_5, y_train_pred)

In [ ]:
y_train_pred[701]

In [ ]:
from sklearn.metrics import precision_score, recall_score
print(precision_score(y_train_5, y_train_pred))
print(recall_score(y_train_5, y_train_pred))

In [ ]:
y_scores = cross_val_predict(sgd_clf, train_data, y_train_5, cv=3,
                                 method="decision_function")

In [ ]:
from sklearn.metrics import precision_recall_curve
precisions, recalls, thresholds = precision_recall_curve(y_train_5, y_scores)
def plot_precision_recall_vs_threshold(precisions, recalls, thresholds):
    plt.plot(thresholds, precisions[:-1], "b--", label="Precision")
    plt.plot(thresholds, recalls[:-1], "g-", label="Recall")
plot_precision_recall_vs_threshold(precisions, recalls, thresholds)
plt.show()

In [ ]:
from sklearn.metrics import roc_curve
fpr, tpr, thresholds = roc_curve(y_train_5, y_scores)
def plot_roc_curve(fpr, tpr, label=None):
    plt.plot(fpr, tpr, linewidth=2, label=label)
    plt.plot([0, 1], [0, 1], 'k--')

plot_roc_curve(fpr, tpr)
plt.show()

In [ ]:
from sklearn.ensemble import RandomForestClassifier
forest_clf = RandomForestClassifier(random_state=42)
y_probas_forest = cross_val_predict(forest_clf, train_data, y_train_5, cv=3,method="predict_proba")

In [ ]:
y_scores_forest = y_probas_forest[:, 1]
fpr_forest, tpr_forest, thresholds_forest = roc_curve(y_train_5,y_scores_forest)


In [ ]:
plt.plot(fpr, tpr, "b:", label="SGD")
plot_roc_curve(fpr_forest, tpr_forest, "Random Forest")
plt.legend(loc="lower right")
plt.show()

In [ ]:
sgd_clf.fit(train_data, train_label)

In [ ]:
print(sgd_clf.predict([train_data[19]]))
sgd_clf.classes_

------------------------------------------------------------------------

In [ ]:
mydict={0: '0', 1: '1', 2: '2', 3: '3', 4: '4', 5: '5', 6: '6', 7: '7', 8: '8', 9: '9'}
mydict[0]